In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv("dallas_weather.csv",index_col='DATE')
df = df[1462:]


df.index = pd.to_datetime(df.index, dayfirst=True)

missing_ratio = df.isnull().sum() / df.shape[0]

for col in df.columns:
    if missing_ratio[col] > 0.3:
        df = df.drop(col, axis=1)

df.isnull().sum() / df.shape[0]
df = df.drop(['STATION','NAME','SNOW','SNWD'], axis=1)


factor_cols = [col for col in df.columns if col.startswith(("WT", "WV"))]
for col in factor_cols:
    prop_ones = (df[col].fillna(0).astype(int) == 1).mean()

    if prop_ones < 0.15:
        df = df.drop(col, axis=1)



cols = ['AWND', 'PRCP','TMAX', 'TMIN', 'WDF2', 'WDF5', 'WSF2', 'WSF5', 'WT01', 'WT16']

df[cols] = df[cols].fillna(df[cols].median())

df.head()


,AWND,PRCP,TMAX,TMIN,WDF2,WDF5,WSF2,WSF5,WT01,WT16
DATE,,,,,,,,,,
1984-01-02,9.84,0.0,47,30,180.0,180.0,21.0,25.1,1,0
1984-01-03,7.38,0.0,58,26,180.0,180.0,21.0,25.1,1,0
1984-01-04,12.53,0.0,68,39,180.0,180.0,21.0,25.1,0,0
1984-01-05,9.17,0.0,72,33,180.0,180.0,21.0,25.1,0,0
1984-01-06,12.30,0.0,65,42,180.0,180.0,21.0,25.1,0,0


In [3]:



def create_safe_features(data):
    data = data.copy()
    data["dayofweek"] = data.index.dayofweek
    data["month"] = data.index.month
    data["dayofyear"] = data.index.dayofyear
    data["dayofmonth"] = data.index.day
    data["weekofyear"] = data.index.isocalendar().week.astype(int)


    data["month_sin"] = np.sin(2 * np.pi * data["month"] / 12)
    data["month_cos"] = np.cos(2 * np.pi * data["month"] / 12)
    data["dayofyear_sin"] = np.sin(2 * np.pi * data["dayofyear"] / 365.25)
    data["dayofyear_cos"] = np.cos(2 * np.pi * data["dayofyear"] / 365.25)


    data["temp_range_lag1"] = (data["TMAX"] - data["TMIN"]).shift(1)
    data["temp_mean_lag1"] = ((data["TMAX"] + data["TMIN"]) / 2).shift(1)

    data["wind_speed_mean_lag1"] = ((data["WSF2"] + data["WSF5"]) / 2).shift(1)
    data["wind_speed_diff_lag1"] = (data["WSF5"] - data["WSF2"]).shift(1)

    wind_dir_diff = abs(data["WDF5"] - data["WDF2"])
    wind_dir_diff = np.minimum(wind_dir_diff, 360 - wind_dir_diff)

    data["wind_dir_diff_lag1"] = wind_dir_diff.shift(1)





    # Lag features: yesterday's weather
    # safe for forecasting tomorrow
    for col in ["TMAX", "TMIN", "PRCP", "AWND", "WSF2", "WSF5"]:
        data[f"{col}_lag1"] = data[col].shift(1)
        data[f"{col}_lag2"] = data[col].shift(2)
        data[f"{col}_lag7"] = data[col].shift(7)
    
    for col in ["TMAX", "TMIN", "PRCP", "AWND"]:
        data[f"{col}_roll3"] = data[col].shift(1).rolling(3).mean()
        data[f"{col}_roll7"] = data[col].shift(1).rolling(7).mean()
        data[f"{col}_roll14"] = data[col].shift(1).rolling(14).mean()

    # -------------------------
    # Rolling extremes
    # -------------------------
    data["TMAX_roll7_max"] = data["TMAX"].shift(1).rolling(7).max()
    data["TMIN_roll7_min"] = data["TMIN"].shift(1).rolling(7).min()
    data["PRCP_roll7_sum"] = data["PRCP"].shift(1).rolling(7).sum()

    # remove rows created by lag/rolling NaNs
    df = data.dropna()
    return df



final_data = create_safe_features(df)


In [4]:
TARGET = "TMAX"
FINAL_TEST_SIZE = 15

final_data = create_safe_features(df)

SAFE_FEATURES = [
    "dayofweek",
    "month",
    "dayofyear",
    "dayofmonth",
    "weekofyear",
    "month_sin",
    "month_cos",
    "dayofyear_sin",
    "dayofyear_cos",

    "TMAX_lag1",
    "TMAX_lag2",
    "TMAX_lag7",
    "TMAX_roll3",
    "TMAX_roll7",
    "TMAX_roll14",
    "TMAX_roll7_max",

    "TMIN_lag1",
    "TMIN_lag2",
    "TMIN_lag7",
    "TMIN_roll3",
    "TMIN_roll7",
    "TMIN_roll14",
    "TMIN_roll7_min",

    "PRCP_lag1",
    "PRCP_lag2",
    "PRCP_lag7",
    "PRCP_roll3",
    "PRCP_roll7",
    "PRCP_roll14",
    "PRCP_roll7_sum",

    "AWND_lag1",
    "AWND_lag2",
    "AWND_lag7",
    "AWND_roll3",
    "AWND_roll7",
    "AWND_roll14",

    "WSF2_lag1",
    "WSF2_lag2",
    "WSF2_lag7",
    "WSF5_lag1",
    "WSF5_lag2",
    "WSF5_lag7",

    "temp_range_lag1",
    "temp_mean_lag1",
    "wind_speed_mean_lag1",
    "wind_speed_diff_lag1",
    "wind_dir_diff_lag1"
]

# Keep only features that actually exist after cleaning/feature engineering
FEATURES = [col for col in SAFE_FEATURES if col in final_data.columns]

# This is the real ML table. It has target + safe lag/rolling features.
model_df = final_data[[TARGET] + FEATURES].dropna().sort_index()



In [5]:
TARGET = "TMAX"
FINAL_TEST_SIZE = 15
N_SPLITS = 6
CV_TEST_SIZE = 15

# Make sure data is sorted by date
model_df = final_data[[TARGET] + FEATURES].dropna().sort_index()

# Split features and target correctly
X = model_df[FEATURES]
y = model_df[TARGET]

# Final 15 days held out for true final testing
# Do NOT use these rows in cross-validation or model selection.
X_train = X.iloc[:-FINAL_TEST_SIZE]
y_train = y.iloc[:-FINAL_TEST_SIZE]

X_test = X.iloc[-FINAL_TEST_SIZE:]
y_test = y.iloc[-FINAL_TEST_SIZE:]

print("Training rows:", len(X_train))
print("Final test rows:", len(X_test))
print("Final test date range:", X_test.index.min(), "to", X_test.index.max())


Training rows: 15430
Final test rows: 15
Final test date range: 2026-04-15 00:00:00 to 2026-04-29 00:00:00


In [6]:
def make_rf_model():
    return RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )


def mean_absolute_percentage_error_safe(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


# Time-series cross-validation on TRAINING DATA ONLY
# Each fold trains on older data and validates on the next block of dates.
tscv = TimeSeriesSplit(
    n_splits=N_SPLITS,
    test_size=CV_TEST_SIZE
)

cv_results = []
cv_predictions = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train), start=1):
    X_fold_train = X_train.iloc[train_idx]
    y_fold_train = y_train.iloc[train_idx]

    X_fold_val = X_train.iloc[val_idx]
    y_fold_val = y_train.iloc[val_idx]

    model = make_rf_model()
    model.fit(X_fold_train, y_fold_train)

    val_pred = model.predict(X_fold_val)

    mae = mean_absolute_error(y_fold_val, val_pred)
    mse = mean_squared_error(y_fold_val, val_pred)
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error_safe(y_fold_val, val_pred)
    r2 = r2_score(y_fold_val, val_pred)

    cv_results.append({
        "fold": fold,
        "train_start": X_fold_train.index.min(),
        "train_end": X_fold_train.index.max(),
        "val_start": X_fold_val.index.min(),
        "val_end": X_fold_val.index.max(),
        "train_rows": len(X_fold_train),
        "val_rows": len(X_fold_val),
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2
    })

    fold_pred_df = pd.DataFrame({
        "fold": fold,
        "actual": y_fold_val,
        "predicted": val_pred,
        "error": y_fold_val - val_pred
    }, index=X_fold_val.index)

    cv_predictions.append(fold_pred_df)

cv_results = pd.DataFrame(cv_results)
cv_predictions = pd.concat(cv_predictions).sort_index()

cv_results


,fold,train_start,train_end,val_start,val_end,train_rows,val_rows,MAE,MSE,RMSE,MAPE,R2
0,1,1984-01-16,2026-01-14,2026-01-15,2026-01-29,15340,15,7.821778,94.030372,9.696926,19.263520,0.354660
1,2,1984-01-16,2026-01-29,2026-01-30,2026-02-13,15355,15,7.577556,90.393032,9.507525,13.243928,0.515221
2,3,1984-01-16,2026-02-13,2026-02-14,2026-02-28,15370,15,5.354667,37.369841,6.113088,7.401851,0.491213
3,4,1984-01-16,2026-02-28,2026-03-01,2026-03-15,15385,15,5.790000,41.253980,6.422926,7.214571,-0.340576
4,5,1984-01-16,2026-03-15,2026-03-16,2026-03-30,15400,15,9.466444,119.504313,10.931803,13.177250,0.267264
5,6,1984-01-16,2026-03-30,2026-03-31,2026-04-14,15415,15,3.740222,20.697315,4.549430,4.644436,0.263615


In [ ]:
print("CV average metrics:")
print(cv_results[["MAE", "MSE", "RMSE", "MAPE", "R2"]].mean())

print("CV metric standard deviations:")
print(cv_results[["MAE", "MSE", "RMSE", "MAPE", "R2"]].std())

# Train final model on all training data, then evaluate once on untouched final 15 days
regressor = make_rf_model()
regressor.fit(X_train, y_train)

y_pred = regressor.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mape = mean_absolute_percentage_error_safe(y_test, y_pred)

print("Final 15-day holdout metrics:")
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R-squared:", r2)

results = pd.DataFrame({
    "actual": y_test,
    "predicted": y_pred,
    "error": y_test - y_pred
}, index=y_test.index)

results


SyntaxError: unterminated string literal (detected at line 19) (827757206.py, line 19)